In [7]:
# plot_loro_7b.py
# 运行方式: python plot_loro_7b.py
# 在 /gemini/code/ 目录下运行

import pickle
import numpy as np
import matplotlib
matplotlib.use('Agg')  # 云服务器无显示器，必须用非交互后端
import matplotlib.pyplot as plt
import os

# ─── 路径配置 ──────────────────────────────────────────────────────
BASE = "/gemini/code"
OUTPUT_DIR = "/gemini/output/figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── 超参数（与 ipynb 保持一致）──────────────────────────────────
hyperparams = {
    "env": "FrozenLake-v1",
    "n_episodes": 150,
    "n_pretrain_eps": 30,
    "n_online_eps": 120,
    "n_exp": 5,
    "smooth": 10,
}

def smooth_curve(data, window):
    """滑动平均平滑曲线"""
    if window <= 1:
        return data
    result = np.copy(data)
    for i in range(len(data)):
        start = max(0, i - window // 2)
        end = min(len(data), i + window // 2 + 1)
        result[i] = np.mean(data[start:end])
    return result

def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f)

# ─── 1. 加载 Qwen 7B 静态数据 ─────────────────────────────────────
print("Loading Qwen 7B dataset...")
path_7b = f"{BASE}/data/FrozenLake_Qwen2.5-7B-Instruct_Neps_200_20260428223255.pkl"
Qwen_7B_dataset = load_pkl(path_7b)

Qwen_7B_rewards = []
for i in range(hyperparams["n_pretrain_eps"]):
    Qwen_7B_rewards.append(Qwen_7B_dataset.episodes[i].compute_return())
Qwen_7B_mean = np.mean(Qwen_7B_rewards)
Qwen_7B_line = np.full(hyperparams["n_episodes"], Qwen_7B_mean)
print(f"  Qwen 7B static reward: {Qwen_7B_mean:.4f}")

# ─── 2. 加载 cache（在线训练结果）────────────────────────────────
# 你已有: cache_FrozenLake_Neps_30.pkl 和 cache_FrozenLake_on_policy_pretrain_exp_rand.pkl
# 尝试从 output/cache 和 code/data 两个位置加载

cache_paths = {
    "cache30": [
        f"{BASE}/data/cache_FrozenLake_Neps_30.pkl",
        "/gemini/output/cache/cache_FrozenLake_Neps_30.pkl",
    ],
    "cache_rand": [
        f"{BASE}/data/cache_FrozenLake_on_policy_pretrain_exp_rand.pkl",
        "/gemini/output/cache/cache_FrozenLake_on_policy_pretrain_exp_rand.pkl",
    ],
    "cache10": [
        f"{BASE}/data/cache_FrozenLake_Neps_10.pkl",
        "/gemini/output/cache/cache_FrozenLake_Neps_10.pkl",
    ],
    "cache20": [
        f"{BASE}/data/cache_FrozenLake_Neps_20.pkl",
        "/gemini/output/cache/cache_FrozenLake_Neps_20.pkl",
    ],
}

loaded = {}
for key, paths in cache_paths.items():
    for p in paths:
        if os.path.exists(p):
            loaded[key] = load_pkl(p)
            print(f"  Loaded {key} from {p}")
            break
    if key not in loaded:
        print(f"  WARNING: {key} not found, will skip related curves")

# ─── 3. 从 cache 提取 7B 在线训练曲线 ────────────────────────────
def extract_returns_from_cache(cache, n_exp, n_episodes, n_pretrain):
    returns_1000 = np.zeros((n_exp, n_episodes))
    returns_3000 = np.zeros((n_exp, n_episodes))

    n_online = n_episodes - n_pretrain

    for i in range(n_exp):

        key_1000 = f"pretrain_{n_pretrain}_eps_1000_steps_{i}"
        key_3000 = f"pretrain_{n_pretrain}_eps_3000_steps_{i}"

        if key_1000 in cache:
            vals = cache[key_1000]
            returns_1000[i][n_pretrain:] = vals[:n_online]

        if key_3000 in cache:
            vals = cache[key_3000]
            returns_3000[i][n_pretrain:] = vals[:n_online]

    return returns_1000, returns_3000

n_eps = hyperparams["n_episodes"]
n_exp = hyperparams["n_exp"]
n_pre = hyperparams["n_pretrain_eps"]  # 30
episodes_x = np.arange(n_eps)

curves = {}  # key -> (mean, std)

# cache30 -> pretrain_30eps
if "cache30" in loaded:
    r1000, r3000 = extract_returns_from_cache(loaded["cache30"], n_exp, n_eps, 30)
    r1000[:, :30] = Qwen_7B_mean  # 前30eps填LLM静态值
    r3000[:, :30] = Qwen_7B_mean
    curves["LORO-7B-1000steps (pre=30)"] = (np.mean(r1000, 0), np.std(r1000, 0))
    curves["LORO-7B-3000steps (pre=30)"] = (np.mean(r3000, 0), np.std(r3000, 0))

if "cache10" in loaded:
    r1000, r3000 = extract_returns_from_cache(loaded["cache10"], n_exp, n_eps, 10)
    r1000[:, :10] = Qwen_7B_mean
    r3000[:, :10] = Qwen_7B_mean
    curves["LORO-7B-1000steps (pre=10)"] = (np.mean(r1000, 0), np.std(r1000, 0))
    curves["LORO-7B-3000steps (pre=10)"] = (np.mean(r3000, 0), np.std(r3000, 0))

if "cache20" in loaded:
    r1000, r3000 = extract_returns_from_cache(loaded["cache20"], n_exp, n_eps, 20)
    r1000[:, :20] = Qwen_7B_mean
    r3000[:, :20] = Qwen_7B_mean
    curves["LORO-7B-1000steps (pre=20)"] = (np.mean(r1000, 0), np.std(r1000, 0))
    curves["LORO-7B-3000steps (pre=20)"] = (np.mean(r3000, 0), np.std(r3000, 0))

# rand pretrain baseline
if "cache_rand" in loaded:
    rand_cache = loaded["cache_rand"]

    rand_1000 = np.zeros((n_exp, n_eps))
    rand_3000 = np.zeros((n_exp, n_eps))

    for i in range(n_exp):

        k1 = f"pretrain_30_eps_1000_steps_{i}_rand"
        k3 = f"pretrain_30_eps_3000_steps_{i}_rand"

        if k1 in rand_cache:
            vals = rand_cache[k1]
            rand_1000[i][30:] = vals[:n_eps - 30]

        if k3 in rand_cache:
            vals = rand_cache[k3]
            rand_3000[i][30:] = vals[:n_eps - 30]

    curves["Random-pretrain-1000steps"] = (
        np.mean(rand_1000, 0),
        np.std(rand_1000, 0),
    )

    curves["Random-pretrain-3000steps"] = (
        np.mean(rand_3000, 0),
        np.std(rand_3000, 0),
    )
# ─── 4. 绘图（参考图风格）────────────────────────────────────────
w = hyperparams["smooth"]

colors = [
    "#d62728",  # 红
    "#2ca02c",  # 绿
    "#9467bd",  # 紫
    "#1f77b4",  # 蓝
    "#ff7f0e",  # 橙
    "#8c564b",  # 棕
    "#e377c2",  # 粉
    "#bcbd22",  # 黄绿
]

fig, ax = plt.subplots(figsize=(10, 5))

# 画 LLM 静态 baseline（水平线）
ax.plot(episodes_x, Qwen_7B_line, color="gray", linestyle="--",
        linewidth=1.5, label="Qwen2.5-7B (static)", zorder=2)

# 画每条在线训练曲线
for idx, (label, (mean, std)) in enumerate(curves.items()):
    color = colors[idx % len(colors)]
    smoothed_mean = smooth_curve(mean, w)
    smoothed_std = smooth_curve(std, w)
    ax.plot(episodes_x, smoothed_mean, color=color, linewidth=1.8,
            label=label, zorder=3)
    ax.fill_between(episodes_x,
                    smoothed_mean - smoothed_std,
                    smoothed_mean + smoothed_std,
                    alpha=0.2, color=color, zorder=1)

# 画预训练/在线分界线（虚线）
ax.axvline(x=n_pre, color="black", linestyle="--", linewidth=1.2, alpha=0.6)
ax.text(n_pre * 0.3, ax.get_ylim()[1] * 0.92,
        "Offline Pre-training", fontsize=9, color="gray")
ax.text(n_pre + (n_eps - n_pre) * 0.3, ax.get_ylim()[1] * 0.92,
        "Online Tuning", fontsize=9, color="gray")

ax.set_xlabel("Episodes", fontsize=12)
ax.set_ylabel("Reward", fontsize=12)
ax.set_title("(a) FrozenLake-v1: LORO with Qwen2.5-7B", fontsize=13)
ax.legend(loc="lower right", fontsize=8, framealpha=0.8)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, n_eps - 1)
ax.set_ylim(bottom=0)

plt.tight_layout()
out_path = f"{OUTPUT_DIR}/FrozenLake_7B_LORO.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"\nFigure saved to: {out_path}")
plt.close()

Loading Qwen 7B dataset...
  Qwen 7B static reward: 0.0000
  Loaded cache30 from /gemini/code/data/cache_FrozenLake_Neps_30.pkl
  Loaded cache_rand from /gemini/code/data/cache_FrozenLake_on_policy_pretrain_exp_rand.pkl
  Loaded cache10 from /gemini/code/data/cache_FrozenLake_Neps_10.pkl
  Loaded cache20 from /gemini/code/data/cache_FrozenLake_Neps_20.pkl

Figure saved to: /gemini/output/figures/FrozenLake_7B_LORO.png


dict_keys(['pretrain_30_eps_1000_steps_0_rand', 'pretrain_30_eps_1000_steps_1_rand', 'pretrain_30_eps_1000_steps_2_rand', 'pretrain_30_eps_1000_steps_3_rand', 'pretrain_30_eps_1000_steps_4_rand', 'pretrain_20_eps_1000_steps_0_rand', 'pretrain_20_eps_1000_steps_1_rand', 'pretrain_20_eps_1000_steps_2_rand', 'pretrain_20_eps_1000_steps_3_rand', 'pretrain_20_eps_1000_steps_4_rand', 'pretrain_10_eps_1000_steps_0_rand', 'pretrain_10_eps_1000_steps_1_rand', 'pretrain_10_eps_1000_steps_2_rand', 'pretrain_10_eps_1000_steps_3_rand', 'pretrain_10_eps_1000_steps_4_rand', 'pretrain_30_eps_3000_steps_0_rand', 'pretrain_30_eps_3000_steps_1_rand', 'pretrain_30_eps_3000_steps_2_rand', 'pretrain_30_eps_3000_steps_3_rand', 'pretrain_30_eps_3000_steps_4_rand', 'pretrain_20_eps_3000_steps_0_rand', 'pretrain_20_eps_3000_steps_1_rand', 'pretrain_20_eps_3000_steps_2_rand', 'pretrain_20_eps_3000_steps_3_rand', 'pretrain_20_eps_3000_steps_4_rand', 'pretrain_10_eps_3000_steps_0_rand', 'pretrain_10_eps_3000_steps